In [20]:
import pandas as pd

df1 = pd.read_json("policy_data/part-00000-tid-7993614477268990145-86e0c334-dc38-4ec8-8d9d-4217b6f7ac05-49801-1-c000.json", lines=True)
df2 = pd.read_json("policy_data/part-00001-tid-7993614477268990145-86e0c334-dc38-4ec8-8d9d-4217b6f7ac05-49802-1-c000.json", lines=True)

df = pd.concat([df1, df2], ignore_index=True)

df.head(20)


,policy_id,customer_id,event_type,event_timestamp,policy_type,premium_amount,coverage_amount,age_of_insured,region
0,POL-58051,CUS-70018,claim,2024-01-21T10:40:41.087Z,"{'type': 'auto', 'brand': 'LifeSecure'}",938.88,6.102073e+04,56.0,North
1,POL-84577,CUS-56010,purchase,2024-07-24T01:56:56.088Z,"{'type': 'life', 'brand': 'InsureCorp'}",687.09,3.132927e+04,62.0,East
2,POL-58297,CUS-4315,purchase,2024-12-22T10:25:58.088Z,"{'type': 'life', 'brand': 'InsureCorp'}",1108.62,2.681022e+04,39.0,West
3,POL-21521,CUS-37602,claim,2024-04-23T23:51:29.088Z,"{'type': 'auto', 'brand': 'LifeSecure'}",1766.21,8.305563e+04,66.0,South
4,POL-69237,CUS-84286,purchase,2024-09-30T03:07:48.088Z,"{'type': 'auto', 'brand': 'InsureCorp'}",905.40,3.844622e+04,71.0,North
5,POL-60927,CUS-39540,purchase,2024-03-25T20:25:51.088Z,"{'type': None, 'brand': 'LifeSecure'}",581.55,2.991422e+04,55.0,North
6,POL-47666,CUS-89696,purchase,2024-08-13T06:52:57.088Z,"{'type': 'health', 'brand': 'LifeSecure'}",1272.79,4.399557e+04,58.0,South
7,POL-16907,CUS-8242,claim,2024-03-16T11:19:41.088Z,"{'type': None, 'brand': 'LifeSecure'}",793.39,4.442296e+04,60.0,North
8,POL-54380,CUS-52665,cancellation,2024-01-08T18:23:04.088Z,"{'type': 'home', 'brand': 'InsureCorp'}",1714.52,1.404010e+04,75.0,West
9,POL-45657,CUS-94911,cancellation,2024-03-27T09:38:29.088Z,"{'type': 'health', 'brand': 'ProtectPlus'}",525.92,4.332432e+04,71.0,West


In [ ]:
df.count()

In [17]:
event_types = df['event_type'].groupby(df['event_type']).size()
event_types


event_type
cancellation    1303
claim           1214
purchase        1209
renewal         1274
Name: event_type, dtype: int64

In [18]:
df['customer_id'].nunique()

3344

In [22]:
df['coverage_amount'].min(), df['coverage_amount'].max(), df['coverage_amount'].mean()

(np.float64(-2147483649.0), np.float64(99988.82), np.float64(-16266706.005828))

In [24]:
df['coverage_amount'][df['coverage_amount'] < 0].count()

np.int64(38)

In [23]:
df['coverage_amount'][df['coverage_amount'] == -2147483649.0].count()

np.int64(38)

In [29]:
df['region'].nunique(), df['region'].value_counts(dropna=False)

(4,
 region
 North    1323
 South    1299
 East     1198
 West     1180
 Name: count, dtype: int64)

In [26]:
df['age_of_insured'].min(), df['age_of_insured'].max(), df['age_of_insured'][df['age_of_insured'] < 0].count(), df['age_of_insured'][df['age_of_insured'] >105].count()

(np.float64(-10.0), np.float64(75.0), np.int64(38), np.int64(0))

In [28]:
import ast

# policy_type is a Python dict string (single quotes, None), so literal_eval rather than json.loads
policy_types = pd.DataFrame(df['policy_type'].apply(ast.literal_eval).tolist())

# dropna=False so the missing product types are counted too
display(policy_types['brand'].value_counts(dropna=False))
display(policy_types['type'].value_counts(dropna=False))

df['policy_type'].nunique(), df['policy_type'].value_counts()

brand
ProtectPlus    1304
InsureCorp     1249
LifeSecure     1233
SafeGuard      1214
Name: count, dtype: int64

type
NaN       2509
health     647
life       622
home       620
auto       602
Name: count, dtype: int64

(20,
 policy_type
 {'type': None, 'brand': 'ProtectPlus'}        651
 {'type': None, 'brand': 'InsureCorp'}         637
 {'type': None, 'brand': 'LifeSecure'}         613
 {'type': None, 'brand': 'SafeGuard'}          608
 {'type': 'health', 'brand': 'ProtectPlus'}    179
 {'type': 'home', 'brand': 'ProtectPlus'}      172
 {'type': 'life', 'brand': 'LifeSecure'}       162
 {'type': 'health', 'brand': 'LifeSecure'}     159
 {'type': 'health', 'brand': 'InsureCorp'}     159
 {'type': 'life', 'brand': 'InsureCorp'}       157
 {'type': 'auto', 'brand': 'InsureCorp'}       157
 {'type': 'home', 'brand': 'SafeGuard'}        157
 {'type': 'life', 'brand': 'ProtectPlus'}      155
 {'type': 'home', 'brand': 'LifeSecure'}       152
 {'type': 'auto', 'brand': 'SafeGuard'}        151
 {'type': 'health', 'brand': 'SafeGuard'}      150
 {'type': 'life', 'brand': 'SafeGuard'}        148
 {'type': 'auto', 'brand': 'LifeSecure'}       147
 {'type': 'auto', 'brand': 'ProtectPlus'}      147
 {'type': 'ho

In [30]:
df["event_timestamp"].min(), df["event_timestamp"].max()

('2024-01-08T18:23:04.088Z', '2025-01-07T12:41:39.187Z')

In [35]:
df["event_date"] = pd.to_datetime(df["event_timestamp"], format="ISO8601", utc=True).dt.date

df["event_date"].value_counts(dropna=False).sort_values(ascending=True).head(20)


event_date
2024-11-28    4
2024-01-08    4
2024-11-08    5
2025-01-07    6
2024-01-26    6
2025-01-05    6
2025-01-02    6
2024-04-09    6
2024-09-14    7
2024-08-31    7
2024-02-22    7
2024-02-26    7
2024-10-20    7
2024-11-19    7
2025-01-01    7
2024-12-19    7
2024-09-29    7
2024-05-15    7
2024-05-10    8
2024-03-05    8
Name: count, dtype: int64